# AI4S Isotope Effect Prediction -- Full Workflow Tutorial

This notebook walks through the complete pipeline:
1. ABACUS DFT data generation
2. DP-GEN active learning
3. DeePMD-kit DPA-2 training
4. i-PI PIMD simulation
5. FEP free energy calculation
6. Alpha prediction and calibration

In [ ]:
import os
import json
import numpy as np
from pathlib import Path

print('Environment ready.')

## 1. ABACUS Template Inspection

Verify the DFT templates are correctly configured with production functionals.

In [ ]:
def inspect_template(molecule):
    tmpl_dir = Path(f'abacus/templates/{molecule}')
    for f in tmpl_dir.glob('*'):
        print(f'--- {f.name} ---')
        print(f.read_text()[:500])
        print()

inspect_template('BF3')

## 2. DP-GEN Configuration

Load and validate the active learning parameters.

In [ ]:
with open('dp_gen/param.json') as f:
    params = json.load(f)

print('Type map:', params['type_map'])
print('Trust bounds: lo={}, hi={}'.format(
    params['model_devi']['trust_lo'],
    params['model_devi']['trust_hi']))
print('Training steps:', params['train']['default_training_param']['training']['numb_steps'])

## 3. DPA-2 Force Field Configuration

Inspect the DPA-2 + DP-LONG training setup.

In [ ]:
with open('deepmd/train/input_dpa2.json') as f:
    dp_config = json.load(f)

print('Model type:', dp_config['model']['descriptor']['type'])
print('Repformer layers:', dp_config['model']['descriptor']['repformer']['nlayers'])
print('Fitting net neurons:', dp_config['model']['fitting_net']['neuron'])
print('DP-LONG enabled:', dp_config['dp_long']['enable'])
print('Start LR:', dp_config['learning_rate']['start_lr'])

## 4. PIMD Beads Convergence

Run the P=32/64/128 convergence test and visualize results.

In [ ]:
import sys
sys.path.insert(0, '.')
from pimd.convergence.test_beads import run_beads_test, plot_beads_convergence
from pathlib import Path

# Run convergence test with mock data
results = run_beads_test(
    beads_values=[32, 64, 128],
    temperature=145.0,
    n_steps_prod=10000,
)

for p, data in results.items():
    qke_mean = np.mean(data['quantum_kinetic'])
    print(f'P={p}: Quantum KE = {qke_mean:.6f} eV')

plot_beads_convergence(results, Path('beads_convergence.png'))
print('Plot saved to beads_convergence.png')

## 5. FEP Free Energy Calculation

Compute DeltaA for 10B -> 11B mass mutation in both phases.

In [ ]:
from pimd.fep.fep_mass_mutation import IsotopePair, run_mass_mutation_fep
from pimd.fep.calculate_delta_A import load_pimd_trajectory, PhaseConfig, compute_delta_delta_a

# Define isotope pair
b_isotopes = IsotopePair(
    element='B',
    mass_light=10.0129,
    mass_heavy=11.0093,
    atom_indices=[0],
)

# Load mock trajectories
liquid_traj = load_pimd_trajectory(Path('mock_liquid.xyz'), n_atoms=4, n_beads=64)
gas_traj = load_pimd_trajectory(Path('mock_gas.xyz'), n_atoms=4, n_beads=64)

# Compute phase-specific DeltaA
liquid_result = run_mass_mutation_fep(liquid_traj, b_isotopes, temperature=145.0, n_beads=64)
gas_result = run_mass_mutation_fep(gas_traj, b_isotopes, temperature=145.0, n_beads=64)

print('Liquid: DeltaA = {:.6f} +/- {:.6f} eV'.format(
    liquid_result['delta_A_eV'], liquid_result['delta_A_std_eV']))
print('Gas:    DeltaA = {:.6f} +/- {:.6f} eV'.format(
    gas_result['delta_A_eV'], gas_result['delta_A_std_eV']))

# Compute differential
dda = compute_delta_delta_a(liquid_result, gas_result)
print('DeltaDeltaA = {:.6f} +/- {:.6f} meV'.format(
    dda['delta_delta_A_meV'], dda['delta_delta_A_std_meV']))

## 6. Alpha Prediction

Convert free energy differential to isotope fractionation factor.

In [ ]:
from pimd.fep.calculate_alpha import (
    calculate_alpha_with_uncertainty,
    classify_isotope_effect,
    format_alpha_report,
    generate_temperature_scan,
)

# Calculate alpha
alpha_result = calculate_alpha_with_uncertainty(
    liquid_result['delta_A_eV'], liquid_result['delta_A_std_eV'],
    gas_result['delta_A_eV'], gas_result['delta_A_std_eV'],
    temperature=145.0,
)

print(format_alpha_report(
    alpha_result['alpha'], alpha_result['alpha_std'],
    145.0, '10B/11B',
))

# Temperature scan
temps, alphas = generate_temperature_scan(
    liquid_result['delta_A_eV'], gas_result['delta_A_eV'],
    t_min=80, t_max=250,
)
print('Alpha at 145 K: {:.6f}'.format(alphas[np.argmin(np.abs(temps - 145.0))]))

## 7. Cross-System Generalization

Adapt the BF3 pipeline to CF4 using LoRA transfer learning.

In [ ]:
import sys
sys.path.insert(0, '.')
from generalization.bf3_to_cf4 import run_cross_system_pipeline
from pathlib import Path

output_dir = Path('transfer_bf3_to_cf4')
run_cross_system_pipeline('BF3', 'CF4', output_dir)

print('\nAdapted configs:')
for f in output_dir.glob('*'):
    print(f'  {f.name}')

## 8. Experimental Calibration

Compare theoretical predictions with industrial measurements.

In [ ]:
from calibration.compare_exp_vs_theory import (
    load_mock_experimental_data,
    compute_agreement_metrics,
    plot_exp_vs_theory,
)

exp_data = load_mock_experimental_data('BF3')
theory_alpha = alphas[np.argmin(np.abs(temps - 145.0))] * np.ones_like(exp_data['temperature'])
theory_std = np.full_like(exp_data['temperature'], 0.0003)

metrics = compute_agreement_metrics(theory_alpha, exp_data['alpha'], exp_data['alpha_std'])
for k, v in metrics.items():
    print(f'{k}: {v}')

plot_exp_vs_theory(exp_data, theory_alpha, theory_std, 'BF3', '10B/11B',
                   Path('exp_vs_theory.png'))